In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import mne
import yaml

with open('config.yaml') as f:
    config = yaml.safe_load(f)

In [ ]:
calibration_data_path  = config["data_calibration"]
fs = config['fs']
trial_duration_s = config['trial_duration_s']
trial_duration_samples = fs * trial_duration_s
total_trials = config['total_trials']
num_chans = config["num_chans"]


example_df = pd.read_csv(f'{calibration_data_path}/S7_eeg_Calibration.csv')

In [ ]:
X = example_df.copy()
y = np.array(example_df["class"])
X.drop(columns = ["TimeStamp","trial","class"], inplace =True)
X = np.array(X)
print(X.shape[0],X.shape[1])

In [ ]:
X_epoched = np.zeros((total_trials, num_chans, trial_duration_samples))

for i_trial in range(total_trials):
    start = i_trial * trial_duration_samples
    end = start + trial_duration_samples
    X_epoched[i_trial,:,:] =  X[start:end,:].T

print(X_epoched.shape[0],X_epoched.shape[1],X_epoched.shape[2])

<h4> CSP

<h4> Covariance matrix for one epoch as an example

In [ ]:
epoch = np.array(X_epoched[0])
epoch_c = epoch - epoch.mean(axis =1, keepdims=True)
print(epoch_c.shape[0],epoch_c.shape[1])

In [ ]:
C = (epoch_c @ epoch_c.T) / (epoch_c.shape[1] - 1)

print("Covariance matrix shape:", C.shape)
print("Diagonal (channel variances):\n", np.diag(C).round(4))

# visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im = axes[0].imshow(C, cmap='RdBu_r', aspect='auto')
axes[0].set_title('Covariance matrix — epoch 0')
axes[0].set_xticks(range(16))
axes[0].set_xticklabels(range(1, 17))
axes[0].set_yticks(range(16))
axes[0].set_yticklabels(range(1, 17))
axes[0].set_xlabel('Channel')
axes[0].set_ylabel('Channel')
plt.colorbar(im, ax=axes[0])

# also look at just the diagonal (per-channel variance)
axes[1].bar(range(16), np.diag(C))
axes[1].set_title('Per-channel variance (diagonal of C)')
axes[1].set_xlabel('Channel')
axes[1].set_ylabel('Variance')
axes[1].set_xticks(range(16))
axes[1].set_xticklabels(range(1, 17))
axes[1].axhline(np.diag(C).mean(), color='red', linestyle='--', label='mean variance')
axes[1].legend()

plt.tight_layout()
plt.show()


<h4> Compute covariance matrices across all trials belonging to their respective classes

In [ ]:
# assume labels is a 1D array of shape (40,) with values 0 and 1
labels = y.copy()
labels[labels == -1] = 0
labels = labels.reshape(total_trials,trial_duration_samples)
print(labels.shape[0],labels.shape[1])


In [ ]:
C_per_class = {}

for cls in [0, 1]:
    trials = X_epoched[labels == cls]          # shape: (n_trials_cls, 16, 2000)
    covs = []
    for trial in trials:
        trial_c = trial - trial.mean(axis=1, keepdims=True)
        C = (trial_c @ trial_c.T) / (trial_c.shape[1] - 1)
        covs.append(C)
    C_per_class[cls] = np.mean(covs, axis=0)   # average over trials

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for cls in [0, 1]:
    im = axes[cls].imshow(C_per_class[cls], cmap='RdBu_r', aspect='auto')
    axes[cls].set_title(f'Average covariance — class {cls}')
    axes[cls].set_xticks(range(16))
    axes[cls].set_xticklabels(range(1, 17))
    axes[cls].set_yticks(range(16))
    axes[cls].set_yticklabels(range(1, 17))
    plt.colorbar(im, ax=axes[cls])

plt.tight_layout()
plt.show()

<h5> calculate difference between covariance matrices

In [ ]:
diff = C_per_class[0] - C_per_class[1]

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(diff, cmap='RdBu_r', aspect='auto')
ax.set_title('Difference (class 0 - class 1)')
ax.set_xticks(range(16))
ax.set_xticklabels(range(1, 17))
ax.set_yticks(range(16))
ax.set_yticklabels(range(1, 17))
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()